# 13 — Preparing Results Data

**Pipeline step:** Section 4 of the paper (Results) — feature engineering shared by every results figure.

**Purpose.**
This notebook is a focused extract of the *setup* cells from `27_Organizando_Resultados.ipynb` (the version **without** "copy"). That notebook mixes these initial calculations together with a long, exploratory tail of plots — most of which were **not** used in the final paper (see the pipeline notes). This notebook keeps only the part that **is** actually used: the feature engineering that both `27_Organizando_Resultados copy.ipynb` (the notebook that produces the paper's figures) and `28.ipynb` depend on, via the `.parquet` file saved at the end.

For each video, computes from its Telegram `occurrences` list:
- **`lifetime`** — how many days the video kept circulating on Telegram (days between its first and last occurrence).
- **`send_count`** — how many times the video was sent/reposted.
- **`transfer_time`** / **`first_appearance`** — how many days after being published on YouTube the video first appeared on Telegram, and the date of that first appearance.

**Input:**
- `data/macrotopics_pred.csv` — Full dataset with macro-topic predictions, from `12_Macrotopic_Classifier.ipynb`.

**Output:**
- `data/not_27.parquet` — the enriched dataset (all original columns + `lifetime`, `send_count`, `transfer_time`, `first_appearance`), read by `27_Organizando_Resultados copy.ipynb` and `28.ipynb`.

> The output filename (`not_27.parquet`) was kept as-is rather than renamed, since renaming it would mean also updating the `pd.read_parquet(...)` call in `27_Organizando_Resultados copy.ipynb` — left untouched to respect the "minimal code changes" approach for notebooks already past the freeze point.

Imports:

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import spearmanr

Loads the full dataset with macro-topic predictions:

In [4]:
path = "data/macrotopics_pred.csv"
df_all = pd.read_csv(path)
df_all.head()

,video_id,text,channel_title,published_at,view_count,like_count,comment_count,occurrences,toxicity,severe_toxicity,...,perspective_insult,perspective_severe_toxicity,perspective_obscene,perspective_toxicity,perspective_identity_attack,perspective_threat,macrotopic_pred,macrotopic_prob,lifetime,send_count
0,jEKzQV5oajY,travel migrant scrap minute,Joe Marsh,2024-04-10T15:05:37Z,1057.0,152.0,32.0,"[{'id': 1416, 'folder': 'channel_1556142220', ...",0.000659,0.000117,...,0.206683,0.021936,0.138744,0.360951,0.367026,0.038566,Others,0.000000,1.0,1
1,xrGGce8cmx8,spiritual warfare charge commit unto thee timo...,WWURD,2023-10-18T14:39:18Z,65.0,6.0,1.0,"[{'id': 60799, 'folder': 'channel_1466271872',...",0.295000,0.001137,...,0.021224,0.005379,0.008414,0.123468,0.049762,0.029415,Religion,0.000000,1.0,1
2,uaozGpSc4nc,putin layer february tucker carlson stand onio...,The Mosaic Ark,2024-02-22T07:30:05Z,330.0,8.0,6.0,"[{'id': 40486, 'folder': 'channel_1235978663',...",0.001735,0.000102,...,0.132458,0.010376,0.003292,0.229804,0.062040,0.033386,Others,0.021935,1.0,4
3,A59ftbhsQUE,GALACTIC ALLIANCE MESSAGE NEWS REMAIN ALERT PR...,GALACTIC ALLIANCE,2024-10-27T04:30:04Z,806.0,136.0,18.0,"[{'id': 449843, 'folder': 'channel_1571505334'...",0.002710,0.000104,...,0.019419,0.005112,0.001093,0.075294,0.017666,0.022509,Religion,0.005000,2.0,5
4,AKU0RokegSo,angeles rain studio city evacuate mudslide atm...,FOX 11 Los Angeles,2024-02-06T01:45:29Z,60863.0,406.0,152.0,"[{'id': 104116, 'folder': 'channel_1438734111'...",0.001515,0.000100,...,0.027841,0.004921,0.002368,0.111507,0.019375,0.028207,Environment,0.000000,2.0,2


`occurrences` is stored as a stringified list of dicts (one per Telegram send event); `safe_literal_eval` parses it back into an actual Python list, treating missing/malformed values as an empty list:

In [ ]:
import ast
import numpy as np 

num_nulls = 0

def safe_literal_eval(value):
    global num_nulls
    if pd.isna(value):
        num_nulls += 1
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, TypeError, AttributeError):
        return []

df_all['occurrences'] = df_all['occurrences'].apply(safe_literal_eval)
print("Número de valores NaN em 'occurrences':", num_nulls)

`calculate_lifetime` — days between a video's first and last Telegram occurrence (1 if it only ever appeared on a single day). Adds the `lifetime` column:

In [ ]:
def calculate_lifetime(occurrence_list):
    dates = [item.get('date') for item in occurrence_list if item.get('date')]
    
    if not dates:
        return None 
    
    datetime_dates = pd.to_datetime(dates)
    min_date = datetime_dates.min()
    max_date = datetime_dates.max()

    if min_date == max_date:
        return 1
    
    diff_days = (max_date - min_date).days + 1 
    return diff_days


df_all['lifetime'] = df_all['occurrences'].apply(calculate_lifetime)
display(df_all[['occurrences', 'lifetime']].head())

,occurrences,lifetime
0,"[{'id': 1416, 'folder': 'channel_1556142220', ...",1.0
1,"[{'id': 60799, 'folder': 'channel_1466271872',...",1.0
2,"[{'id': 40486, 'folder': 'channel_1235978663',...",1.0
3,"[{'id': 449843, 'folder': 'channel_1571505334'...",2.0
4,"[{'id': 104116, 'folder': 'channel_1438734111'...",2.0


`contar_envios` — number of Telegram occurrences for a video. Adds the `send_count` column:

In [ ]:
def contar_envios(valor):
    try:
        if isinstance(valor, list):
            return len(valor)
        
        if isinstance(valor, str):
            lista_real = ast.literal_eval(valor)
            return len(lista_real)
            
        return None 
    except:
        return None

df_all['send_count'] = df_all['occurrences'].apply(contar_envios)
df_all[['occurrences', 'send_count']].head()

,occurrences,send_count
0,"[{'id': 1416, 'folder': 'channel_1556142220', ...",1
1,"[{'id': 60799, 'folder': 'channel_1466271872',...",1
2,"[{'id': 40486, 'folder': 'channel_1235978663',...",4
3,"[{'id': 449843, 'folder': 'channel_1571505334'...",5
4,"[{'id': 104116, 'folder': 'channel_1438734111'...",2


`calculate_transfer_time` — days between the video's YouTube publish date (`published_at`) and its first appearance on Telegram (0 if it appeared before or on the same day it was published, e.g. due to timezone rounding). Adds `transfer_time` and `first_appearance`:

In [ ]:
def calculate_transfer_time(row):
    occurrences = row['occurrences']

    if not occurrences:
        return None

    first_appearance_str = min(occurrences, key=lambda x: x['date'])['date']

    first_date = pd.to_datetime(first_appearance_str).tz_localize(None)
    published_at = pd.to_datetime(row['published_at']).tz_localize(None)

    diff = (first_date - published_at).days
    return max(0, diff)

df_all['transfer_time'] = df_all.apply(calculate_transfer_time, axis=1)
df_all['first_appearance'] = df_all['occurrences'].apply(lambda occ: min(occ, key=lambda x: x['date'])['date'] if occ else None)

Saves the enriched dataset — this is the file `27_Organizando_Resultados copy.ipynb` and `28.ipynb` load as their starting point:

In [ ]:
df_all.to_parquet("data/not_27.parquet", index=False)